# Building Multimodal RAG Application #4: Video Preprocessing

This Colab-ready companion notebook implements the article's three preprocessing paths: align frames with an existing WebVTT transcript, generate a transcript with Whisper, and caption silent-video frames with an open vision-language model. Expensive model steps are optional and clearly separated.

In [ ]:
!pip -q install opencv-python-headless webvtt-py moviepy openai-whisper transformers accelerate

In [ ]:
from pathlib import Path
import json
import requests

import cv2
from PIL import Image
import webvtt

## Download a small example video

Replace this public sample with a video you are authorized to process.

In [ ]:
DATA_DIR = Path("multimodal_rag_video")
FRAMES_DIR = DATA_DIR / "frames"
DATA_DIR.mkdir(exist_ok=True)
FRAMES_DIR.mkdir(exist_ok=True)

VIDEO_URL = (
    "https://multimedia-commons.s3-us-west-2.amazonaws.com/"
    "data/videos/mp4/010/a07/010a074acb1975c4d6d6e43c1faeb8.mp4"
)
VIDEO_PATH = DATA_DIR / "sample.mp4"
if not VIDEO_PATH.exists() or VIDEO_PATH.stat().st_size < 1024:
    with requests.get(VIDEO_URL, stream=True, timeout=120) as response:
        response.raise_for_status()
        with VIDEO_PATH.open("wb") as video_file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                video_file.write(chunk)
VIDEO_PATH

## Sample frames and preserve timestamps

In [ ]:
def extract_frames(video_path: Path, output_dir: Path, every_seconds: float = 10.0):
    capture = cv2.VideoCapture(str(video_path))
    fps = capture.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        raise ValueError("Could not determine the video's frame rate.")

    hop = max(1, round(fps * every_seconds))
    records = []
    frame_number = 0
    sample_number = 0

    while True:
        success, frame = capture.read()
        if not success:
            break
        if frame_number % hop == 0:
            timestamp_ms = capture.get(cv2.CAP_PROP_POS_MSEC)
            frame_path = output_dir / f"frame_{sample_number:04d}.jpg"
            cv2.imwrite(str(frame_path), frame)
            records.append({
                "frame_path": str(frame_path),
                "timestamp_ms": round(timestamp_ms),
                "video_path": str(video_path),
            })
            sample_number += 1
        frame_number += 1

    capture.release()
    return records

frame_records = extract_frames(VIDEO_PATH, FRAMES_DIR, every_seconds=10)
frame_records[:3]

## Path A: align frames with an existing WebVTT transcript

Place a `captions.vtt` file in `DATA_DIR`, then run this cell. Each frame receives the caption active at its timestamp.

In [ ]:
def vtt_time_to_ms(value: str) -> float:
    hours, minutes, seconds = map(float, value.split(":"))
    return (hours * 3600 + minutes * 60 + seconds) * 1000

def align_vtt(records, vtt_path: Path):
    cues = list(webvtt.read(str(vtt_path)))
    aligned = []
    for record in records:
        timestamp = record["timestamp_ms"]
        matching = [
            cue.text.replace("\n", " ")
            for cue in cues
            if vtt_time_to_ms(cue.start) <= timestamp <= vtt_time_to_ms(cue.end)
        ]
        aligned.append({**record, "transcript": " ".join(matching)})
    return aligned

VTT_PATH = DATA_DIR / "captions.vtt"
if VTT_PATH.exists():
    frame_records = align_vtt(frame_records, VTT_PATH)

## Path B: transcribe spoken video with Whisper (optional)

This model step can take several minutes. A GPU runtime is recommended. Whisper returns timestamped segments that can be aligned with the sampled frames using the same interval logic as the WebVTT path.

In [ ]:
RUN_WHISPER = False

if RUN_WHISPER:
    import whisper

    whisper_model = whisper.load_model("small")
    transcription = whisper_model.transcribe(str(VIDEO_PATH), language="en")
    segments = transcription["segments"]
    for record in frame_records:
        second = record["timestamp_ms"] / 1000
        record["transcript"] = " ".join(
            segment["text"].strip()
            for segment in segments
            if segment["start"] <= second <= segment["end"]
        )

## Path C: caption silent-video frames with BLIP (optional)

The archived article used a hosted LLaVA helper. This current notebook keeps the same frame-captioning boundary but uses the public `Salesforce/blip-image-captioning-base` checkpoint directly. You can replace it with any LVLM client.

In [ ]:
RUN_VISUAL_CAPTIONING = False

if RUN_VISUAL_CAPTIONING:
    from transformers import BlipForConditionalGeneration, BlipProcessor

    checkpoint = "Salesforce/blip-image-captioning-base"
    processor = BlipProcessor.from_pretrained(checkpoint)
    caption_model = BlipForConditionalGeneration.from_pretrained(checkpoint)

    for record in frame_records:
        frame = Image.open(record["frame_path"]).convert("RGB")
        inputs = processor(images=frame, return_tensors="pt")
        generated = caption_model.generate(**inputs, max_new_tokens=40)
        record["visual_caption"] = processor.decode(
            generated[0], skip_special_tokens=True
        )

## Save retrieval-ready metadata

In [ ]:
METADATA_PATH = DATA_DIR / "frame_metadata.json"
METADATA_PATH.write_text(json.dumps(frame_records, indent=2))
print(f"Saved {len(frame_records)} records to {METADATA_PATH}")
frame_records[:2]

## Next step

Each metadata record now links a source video and timestamp to a sampled frame plus its transcript or visual caption. Part 5 will embed and index these records for multimodal retrieval.